# Notebook 24 — Balanced Panel Stability Check

Verifies the 2020->2021 gap swing is not a lender-composition artefact.

**Input:** panel files  **Output:** table_24  **Runtime:** ~15 min

In [1]:

import pandas as pd, numpy as np
from scipy import stats
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
PROC=Path("../data/processed"); TABS=Path("../outputs/tables"); TABS.mkdir(exist_ok=True)
YEARS=[2020,2021,2022,2023,2024]; BLACK_CODE=3; MIN_B=10; MIN_W=10; MAX_PER=250
print("NB24: BALANCED PANEL STABILITY CHECK")
print("Tests whether the 2020->2021 gap swing is a composition artefact")
lender_sets={}; dfs={}
for yr in YEARS:
    fp=PROC/f"panel_{yr}.csv"
    if not fp.exists(): print(f"  {yr}: missing"); continue
    df=pd.read_csv(fp,usecols=["lei","applicant_race_1","approved","income","loan_amount","ltv"])
    df=df[df["applicant_race_1"].isin([BLACK_CODE,5])].copy()
    df["black"]=(df["applicant_race_1"]==BLACK_CODE).astype(int)
    df["approved"]=pd.to_numeric(df["approved"],errors="coerce")
    df["income"]  =pd.to_numeric(df["income"],  errors="coerce")
    df["ltv"]     =pd.to_numeric(df["ltv"],     errors="coerce")
    df=df.dropna(subset=["approved","income","ltv"])
    lr=df.groupby("lei")["black"].agg(["sum","count"])
    valid=lr[(lr["sum"]>=MIN_B)&(lr["count"]-lr["sum"]>=MIN_W)].index
    df=df[df["lei"].isin(valid)]
    lender_sets[yr]=set(df["lei"].unique()); dfs[yr]=df
    print(f"  {yr}: {df['lei'].nunique():,} lenders")
print()
yrs=sorted(lender_sets.keys())
print("YEAR-PAIR OVERLAP:")
for i in range(len(yrs)-1):
    y1,y2=yrs[i],yrs[i+1]
    s1,s2=lender_sets[y1],lender_sets[y2]
    pct=len(s1&s2)/min(len(s1),len(s2))*100
    print(f"  {y1}-{y2}: {len(s1&s2):,} shared  ({pct:.1f}% overlap)")
balanced = set.intersection(*[lender_sets[y] for y in yrs if y in lender_sets])
print(f"Balanced panel (all years): {len(balanced):,} lenders")


NB24: BALANCED PANEL STABILITY CHECK
Tests whether the 2020->2021 gap swing is a composition artefact
  2020: 1,861 lenders
  2021: 1,963 lenders
  2022: 1,910 lenders
  2023: 1,730 lenders
  2024: 1,632 lenders

YEAR-PAIR OVERLAP:
  2020-2021: 1,707 shared  (91.7% overlap)
  2021-2022: 1,716 shared  (89.8% overlap)
  2022-2023: 1,607 shared  (92.9% overlap)
  2023-2024: 1,505 shared  (92.2% overlap)
Balanced panel (all years): 1,253 lenders


In [2]:

def gap_fe(df_input, max_per=MAX_PER):
    df=df_input.copy()
    def cap(g):
        b=g[g["black"]==1]; w=g[g["black"]==0]
        return pd.concat([b.sample(min(len(b),max_per),random_state=42),
                          w.sample(min(len(w),max_per),random_state=42)])
    df=df.groupby("lei",group_keys=False).apply(cap)
    df=df.dropna(subset=["approved","income","ltv"])
    regs=["black","income","loan_amount","ltv"]
    lm=df.groupby("lei")[["approved"]+regs].transform("mean")
    for c in ["approved"]+regs: df[c+"_dm"]=df[c]-lm[c]
    Xc=[c+"_dm" for c in regs]; dr=df[["approved_dm"]+Xc+["lei"]].dropna()
    X=dr[Xc].values; y=dr["approved_dm"].values; lei=dr["lei"].values
    Xf=np.column_stack([np.ones(len(X)),X]); coef,_,_,_=np.linalg.lstsq(Xf,y,rcond=None)
    e=y-Xf@coef; ul=np.unique(lei); G=len(ul); n=len(y); k=Xf.shape[1]
    if G<2: return None
    adj=(G/(G-1))*((n-1)/(n-k)); br=np.linalg.inv(Xf.T@Xf); mt=np.zeros((k,k))
    for lend in ul:
        idx=(lei==lend); sc=Xf[idx].T@e[idx]; mt+=np.outer(sc,sc)
    vcov=adj*br@mt@br; se=np.sqrt(np.diag(vcov))
    cn=["const"]+Xc; bi=cn.index("black_dm")
    beta=coef[bi]*100; bse=se[bi]*100; ts=beta/bse if bse>0 else 0
    p=2*(1-stats.t.cdf(abs(ts),df=G-1))
    return {"N_obs":n,"N_lenders":G,"Beta_pp":round(beta,3),"SE":round(bse,3),
            "T_stat":round(ts,3),"P_value":round(p,4)}

print("="*60)
print("FULL vs BALANCED PANEL — YEAR BY YEAR")
print("="*60)
results=[]
for yr in yrs:
    if yr not in dfs: continue
    rf = gap_fe(dfs[yr])
    df_b = dfs[yr][dfs[yr]["lei"].isin(balanced)]
    rb = gap_fe(df_b) if len(df_b)>0 else None
    row={"Year":yr}
    if rf: row.update({"Full_Beta":rf["Beta_pp"],"Full_SE":rf["SE"],"Full_N":rf["N_lenders"]})
    if rb: row.update({"Bal_Beta":rb["Beta_pp"], "Bal_SE":rb["SE"], "Bal_N":rb["N_lenders"]})
    results.append(row)
    fs = f"Full: {rf['Beta_pp']:+.2f}pp (N={rf['N_lenders']:,})" if rf else ""
    bs = f"Bal:  {rb['Beta_pp']:+.2f}pp (N={rb['N_lenders']:,})" if rb else ""
    print(f"  {yr}:  {fs}   {bs}")

df_res=pd.DataFrame(results)
df_res.to_csv(TABS/"table_24_balanced_panel.csv",index=False)
print("Saved table_24_balanced_panel.csv")

if "Full_Beta" in df_res.columns and "Bal_Beta" in df_res.columns:
    diff = (df_res["Full_Beta"]-df_res["Bal_Beta"]).abs().max()
    print(f"Max abs difference full vs balanced: {diff:.2f}pp")
    if diff<1.0:
        print("RESULT: Stable across composition. Gap swing reflects genuine")
        print("  approval rate changes, not lender-composition artefacts.")
        print("  Add 1 sentence to data section confirming this.")
    else:
        print("NOTE: Some composition sensitivity. Report balanced panel")
        print("  as robustness check in appendix.")
print("NB24 COMPLETE")


FULL vs BALANCED PANEL — YEAR BY YEAR
  2020:  Full: -9.64pp (N=1,861)   Bal:  -10.22pp (N=1,253)
  2021:  Full: -9.21pp (N=1,963)   Bal:  -9.61pp (N=1,253)
  2022:  Full: -10.81pp (N=1,910)   Bal:  -11.04pp (N=1,253)
  2023:  Full: -11.44pp (N=1,730)   Bal:  -11.72pp (N=1,253)
  2024:  Full: -11.30pp (N=1,632)   Bal:  -11.60pp (N=1,253)
Saved table_24_balanced_panel.csv
Max abs difference full vs balanced: 0.58pp
RESULT: Stable across composition. Gap swing reflects genuine
  approval rate changes, not lender-composition artefacts.
  Add 1 sentence to data section confirming this.
NB24 COMPLETE
